# 🧠 Homework: Building a Transformer Language Model from Scratch

In this assignment, you will build, train, and decode from a Transformer-based language model using the **Tiny Shakespeare** dataset. This is a character-level language modeling task designed to help you understand the inner workings of autoregressive text generation using the Transformer architecture.

---

## 📚 Dataset

We use the **Tiny Shakespeare** dataset, a ~1MB text file containing the works of William Shakespeare. It is widely used for testing small language models. In this assignment, we’ll truncate it to just the first 100,000 characters to ensure fast training on CPU or GPU.

The text will be encoded as a sequence of characters, with each unique character assigned an integer index. This turns our modeling task into a classic sequence prediction problem: given a sequence of previous characters, predict the next one.

---

## 🎯 Objectives

You will implement and train a Transformer language model with the following goals:

1. **Understand Transformer language models**  
   You will fill in key parts of a Transformer encoder-based autoregressive language model and use it to predict the next character in a sequence.

2. **Train using maximum likelihood**  
   Learn how to compute the loss using cross-entropy and update model parameters to minimize it.

3. **Generate text using decoding algorithms**  
   You'll implement and experiment with two popular decoding strategies:
   - **Top-k sampling** with temperature
   - **Beam search**

4. **Analyze and compare decoding results**  
   After training, you'll generate Shakespeare-like text and observe the differences in output quality between greedy decoding, sampling, and beam search.

---

## 🛠️ Skills You'll Practice

- Text preprocessing and tokenization
- Embedding layers and positional encoding
- Building TransformerEncoder with PyTorch
- Applying causal masks during decoding
- Understanding softmax sampling vs deterministic decoding
- Implementing decoding algorithms from scratch
- Autoregressive generation and recursive input expansion

---

## 📝 What To Do

- Read the starter code carefully.
- Complete the `TODO` sections marked in the model definition, training loop, and decoding functions.
- Run the training loop and generate outputs using both decoding methods.
- Observe, compare, and interpret the generated samples.

---

## 🚨 Guidelines

- **Do not modify** the structure or logic of the provided code — your task is to understand and complete it.
- Add inline comments to clarify your understanding if needed.
- Make sure your generated outputs make sense, and feel free to tweak decoding parameters (`top_k`, `temperature`, `beam_width`) to see how they affect results.

---

By the end of this homework, you will have a solid foundation for understanding how GPT-like models are built and used in practice.

Good luck and have fun!

## Problem 1:  Transformer Language Model (15 points)

In [3]:
# %%
# # Homework: Training a Transformer Language Model on Tiny Shakespeare
#
# In this assignment, you will build a character-level language model using the Transformer architecture.
# You'll train it on the Tiny Shakespeare dataset and experiment with two decoding strategies:
# top-k sampling and beam search.
#
# Your job is to fill in the TODO sections — these are deliberately left for you to implement.
# You will learn:
# - How to encode text for language modeling
# - How Transformer-based autoregressive models work
# - How to train a language model using cross-entropy loss
# - How to decode text using top-k sampling and beam search
#
# DO NOT change the structure or logic of the code. Your task is to understand and complete the missing parts.

import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

install_if_missing("torch")
install_if_missing("tqdm")
install_if_missing("datasets")

# %%
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import math
from datasets import load_dataset

# Set a manual seed for reproducibility
torch.manual_seed(42)

# %%
# Load and preprocess the Tiny Shakespeare dataset
import urllib.request

_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

with urllib.request.urlopen(_url) as _response:

    text_data = _response.read().decode("utf-8")

text = text_data[:100000]

# -------------------------------------
# Character-level tokenization
# -------------------------------------
# We treat each individual character as a token. So we need to:
# 1. Identify all unique characters in the dataset (our vocabulary)
# 2. Assign a unique integer (index) to each character
# 3. Define functions to convert between characters and indices

# Step 1: extract all unique characters from the dataset and sort them
chars = sorted(list(set(text)))

# Step 2: get vocabulary size = total number of unique characters
vocab_size = len(chars)

# Step 3: create two lookup dictionaries
# stoi = string to index (e.g. 'a' → 0, 'b' → 1, etc.)
# itos = index to string (e.g. 0 → 'a', 1 → 'b', etc.)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# Step 4: define helper functions to convert between strings and token sequences
# Example:
# encode("hello") → [34, 15, 26, 26, 41]
# decode([34, 15, 26, 26, 41]) → "hello"
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Now encode the full text as a long tensor of integers
# This tensor will be used to train the language model
data = torch.tensor(encode(text), dtype=torch.long)  # shape: (100000,)

# -------------------------------------
# Train/validation split
# -------------------------------------
# We split the data into two parts:
# - 90% for training
# - 10% for validation (to monitor generalization)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

# %%
# Transformer language model definition
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, block_size=128):
        super().__init__()
        # TODO:
        # Define the model components:
        # 1. Token embedding layer: turns input indices into vectors of size d_model.
        # 2. Positional embedding: learnable positional encodings to add order info.
        # 3. Transformer Encoder: built using nn.TransformerEncoder and a base encoder layer.
        # 4. lm_head: a final linear layer mapping from d_model to vocab_size.
        # 5. Save the block_size so we can use it to slice the positional embeddings.
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.block_size = block_size


    def forward(self, x):
        # TODO:
        # 1. Add token + position embeddings
        #    shape: (B, T, C)
        embed_token = self.token_embedding(x) 
        B, T, C = embed_token.shape
        positions = torch.arange(T, device=x.device)
        embed_pos = self.pos_embedding(positions)
        embed_pos = embed_pos.unsqueeze(0)
        
        # 2. Transformer expects input of shape (T, B, C), so we need to permute.
        h = embed_token + embed_pos  
        h = h.permute(1, 0, 2)
        # 3. Build a causal mask so that token i cannot attend to token j > i.
        #    Hint: Use `torch.triu(torch.ones(T, T), diagonal=1)` to create an upper triangular mask.
        #    Then convert to boolean and pass to transformer as `mask=...`
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
               

        # 4. Pass through transformer.
        out = self.transformer_encoder(h, mask)

        # 5. Permute back to (B, T, C)
        out = out.permute(1, 0, 2)

        # 6. Project to logits with lm_head
        out = self.lm_head(out)
        return out



# %%
# Prepare training batches
block_size = 128  # sequence length
batch_size = 16   # how many sequences per batch

def get_batch(split):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x, y

# %%
# Training loop
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TransformerLM(vocab_size=vocab_size, block_size=block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

training_steps = 2000
total_loss = 0
for step in tqdm(range(training_steps)):
    x, y = get_batch("train")
    x, y = x.to(device), y.to(device)

    # TODO:
    # 1. Forward pass through the model to get logits.
    #    define loss = F.cross_entroy
    logits = model(x)
    B, T, V = logits.shape # B = batch dimension, T = time / sequence position dimension, V = vocabulary dimension
    
    # 2. Compute the loss: cross_entropy between logits and targets.
    #    - Hint: reshape logits to (B*T, vocab_size), y to (B*T)
    loss = F.cross_entropy(logits.reshape(B * T, V), y.reshape(B * T))
    total_loss += loss.item()
    # 3. Zero the gradients, backprop, and optimizer step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



    if step % 200 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

/opt/conda/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
  0%|          | 1/2000 [00:00<20:29,  1.63it/s]

Step 0: loss = 4.2097


 10%|█         | 201/2000 [02:06<21:04,  1.42it/s]

Step 200: loss = 2.3778


 20%|██        | 401/2000 [04:21<16:26,  1.62it/s]

...

 80%|████████  | 1601/2000 [17:09<04:20,  1.53it/s]

Step 1600: loss = 1.3910


 90%|█████████ | 1801/2000 [19:28<02:30,  1.33it/s]

Step 1800: loss = 1.3250


100%|██████████| 2000/2000 [21:52<00:00,  1.52it/s]


## Problem 2:  top_k_sample (5 points)

In [4]:
# %%
# ---------------------------------------------
# Top-k sampling with temperature
# ---------------------------------------------
# This function selects the next token to generate based on:
# - Limiting possible tokens to only the top-k most probable ones
# - Using temperature to scale the distribution and control randomness
# - Sampling from the resulting softmax distribution
import heapq

def top_k_sample(logits, k=5, temperature=1.0):
    # TODO:
    # Step 1: Adjust the logits by dividing by temperature
    #         - temperature < 1 → makes the distribution sharper (less random)
    #         - temperature > 1 → flattens the distribution (more random)
    # Step 2: Get the top-k logits and their corresponding token indices
    #         - Hint: use torch.topk to retrieve top-k values and indices
    # Step 3: Convert the top-k logits into a probability distribution using softmax
    # Step 4: Sample one token index from the top-k using torch.multinomial
    #         - torch.multinomial draws samples based on the probabilities
    adjusted = logits / temperature
    topk_values, topk_indices = torch.topk(adjusted, k, dim=1)
    probs = F.softmax(topk_values, dim=-1)
    sample = torch.multinomial(probs, 1)
    next_token = torch.gather(topk_indices, -1, sample)
    return next_token
    

# %%
# ---------------------------------------------
# Autoregressive text generation using top-k sampling
# ---------------------------------------------
# This function generates text one token at a time.
# At each step:
# - Feed the current sequence into the model
# - Extract the logits for the last token position
# - Use top_k_sample() to choose the next token
# - Append the new token to the context
# - Repeat

def generate(model, prompt, max_new_tokens=200, top_k=10, temperature=0.8):
    model.eval()

    # Encode the input prompt into token IDs, shape: (1, T)
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # TODO:
        # Step 1: If context is longer than block size, truncate to the most recent block_size tokens
        #         - Transformers are limited to fixed-length input windows
        context_cond = context[:, -model.block_size:]
        # Step 2: Disable gradient tracking with torch.no_grad() for inference
        # Step 3: Feed context into model to get logits (shape: [B, T, vocab_size])
        with torch.no_grad():
            logits = model(context_cond)
        # Step 4: Get the logits for the last time step: logits[:, -1, :]
        #         - This corresponds to the predicted distribution for the next token
        last_step = logits[:, -1, :]
        # Step 5: Use top_k_sample() to get the next token ID
        next_token = top_k_sample(last_step, top_k, temperature).view(1, 1)
        # Step 6: Append that new token to the context using torch.cat
        context = torch.cat([context, next_token], dim=1)
    return decode(context[0].tolist())


# Call the generator and print result
print("\nGenerated with top-k sampling:")
print(generate(model, "ROMEO:", top_k=10, temperature=0.5))


Generated with top-k sampling:
ROMEO:
Look, sir, you shall be consul!

CORIOLANUS:
Hear me was a great it be to the people,
Which he would see him be like us the breath.

BRUTUS:
Why, if when we they are newer
To the came of your his for


## Problem 3:  Beam Search (5 points)

In [20]:
# %%
# ------------------------------------------------
# Beam Search Generation
# ------------------------------------------------
# Beam search keeps track of the top `beam_width` highest-probability sequences.
# At each decoding step, it expands each current sequence with its top-k most likely next tokens,
# computes the cumulative log-probability of each expanded candidate, and selects the top `beam_width` overall.

def beam_search(model, prompt, max_new_tokens=100, beam_width=3):
    model.eval()

    # Start with the prompt as the initial sequence and a score of 0
    sequences = [(torch.tensor(encode(prompt), device=device).unsqueeze(0), 0)]

    for _ in range(max_new_tokens):

        # TODO:
        # Your goal is to expand each of the current sequences in the beam
        # and keep only the top `beam_width` sequences by total score.
        #
        # Hints:
        # - Create a new list called `all_candidates` to collect all possible next-step sequences.
        # - Loop through each sequence in the current beam:
        #     - Truncate the sequence if it's longer than the model's `block_size`.
        #     - Pass the sequence into the model using `with torch.no_grad()` to get logits.
        #     - Extract the logits for the last token (logits[0, -1]).
        #     - Convert logits to log-probabilities using `F.log_softmax(...)`.
        #     - Use `torch.topk(...)` to get the top `beam_width` tokens and their scores.
        #     - For each of these top tokens:
        #         - Create a new candidate sequence by appending the token.
        #         - Add the token's log-probability to the sequence's total score.
        #         - Append the tuple (new_sequence, new_score) to `all_candidates`.
        # - After generating all candidate sequences, sort them by score (highest first).
        # - Keep only the top `beam_width` sequences for the next round.
        # - Repeat until you've generated `max_new_tokens` tokens.
        #
        # Pseudocode:
        #   for each seq, score in sequences:
        #       logits = model(seq)
        #       next_token_logits = logits[0, -1]
        #       log_probs = log_softmax(next_token_logits)
        #       topk_probs, topk_indices = topk(log_probs, beam_width)
        #       for i in range(beam_width):
        #           new_seq = seq + [topk_indices[i]]
        #           new_score = score + topk_probs[i]
        #           all_candidates.append((new_seq, new_score))
        #   sequences = top-k all_candidates by score
        all_candidates = []
        for seq, score in sequences: 
            seq_cond = seq[:, -model.block_size:]
            with torch.no_grad():
                logits = model(seq_cond)
            next_token_logits = logits[:, -1, :]
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            topk_probs, topk_indices = torch.topk(log_probs, beam_width)
            for i in range(beam_width):
                next_token = topk_indices[0, i].view(1, 1)
                new_seq = torch.cat([seq, next_token], dim=1)
                new_score = score + topk_probs[0, i].item()
                all_candidates.append((new_seq, new_score))
        sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        
    
    best_seq, best_score = sequences[0]
    return decode(best_seq[0].tolist())
        
        


# Run beam search generation and print the result
print("\nGenerated with beam search:")
print(beam_search(model, "ROMEO:", beam_width=4))


Generated with beam search:
ROMEO:
Look, good madam.

VOLUMNIA:
I will be consul!

VOLUMNIA:
Hear your voices!

VOLUMNIA:
Hear madam; 
